# Export RS3 train, test, and unseen sequences

Place this notebook in the repository's `code` directory.

It recreates the fixed seed-42 split and exports aligned TXT files for:

- 20-nt sgRNA spacer sequence
- 20-nt target protospacer
- full target-context sequence, normally 30 nt
- PAM sequence
- activity value


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from datasets import dataset_list

SPLIT_SEED = 42
PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../models/rs3_fixed_split_100_trials/split_sequences")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_NAMES_FILE = PROCESSED_DIR / "train_data_names.csv"
assert TRAIN_NAMES_FILE.exists(), f"Missing: {TRAIN_NAMES_FILE.resolve()}"

print("Output directory:", OUTPUT_DIR.resolve())


Output directory: /Users/zhangjiongyu/rs_dev/models/rs3_fixed_split_100_trials/split_sequences


In [3]:
train_data_names = (
    pd.read_csv(TRAIN_NAMES_FILE)["name"]
    .dropna()
    .astype(str)
    .tolist()
)

train_data_list = [ds for ds in dataset_list if ds.name in train_data_names]

print("Datasets:", [ds.name for ds in train_data_list])

for ds in train_data_list:
    ds.load_data()
    ds.set_sgrnas()


Datasets: ['Doench2014_mouse', 'Doench2014_human', 'Doench2016', 'Kim2019_train', 'Wang2014', 'Xiang2021', 'Munoz2016']


In [5]:
sg_df_list = []

for ds in train_data_list:
    sg_df = ds.get_sg_df(include_group=True, include_activity=True).copy()
    sg_df["dataset"] = ds.name
    sg_df["tracr"] = ds.tracr
    sg_df_list.append(sg_df)

sg_df_groups = (
    pd.concat(sg_df_list, ignore_index=True)
    .groupby("sgRNA Context Sequence", as_index=False)
    .agg(
        target=(
            "sgRNA Target",
            lambda x: ", ".join(
                sorted({
                    str(v).upper()
                    for v in x
                    if not pd.isna(v) and str(v).strip() != ""
                })
            ),
        )
    )
)

sg_df_groups["target"] = sg_df_groups.apply(
    lambda row: row["target"]
    if row["target"] != ""
    else row["sgRNA Context Sequence"],
    axis=1,
)

all_data = (
    pd.concat(sg_df_list, ignore_index=True)
    .merge(
        sg_df_groups[["sgRNA Context Sequence", "target"]],
        on="sgRNA Context Sequence",
        how="inner",
    )
    .sort_values(["dataset", "target"])
    .reset_index(drop=True)
)

all_data["sgRNA Activity"] = pd.to_numeric(
    all_data["sgRNA Activity"], errors="coerce"
)

all_data = all_data.dropna(
    subset=[
        "sgRNA Sequence",
        "sgRNA Context Sequence",
        "sgRNA Activity",
        "dataset",
        "target",
    ]
).reset_index(drop=True)

print("Combined rows:", len(all_data))
print("Unique target groups:", all_data["target"].nunique())
display(all_data.head())


Combined rows: 51087
Unique target groups: 20637


,sgRNA Sequence,sgRNA Context Sequence,PAM Sequence,sgRNA Target,sgRNA Activity,dataset,tracr,target
0,AAATAATACCAACAACTGGA,TCAGAAATAATACCAACAACTGGAGGGAGA,GGG,ANPEP,0.947185,Doench2014_human,Hsu2013,ANPEP
1,AACAGCTCACTGATCTGGGC,GTCAAACAGCTCACTGATCTGGGCCGGCGT,CGG,ANPEP,0.786662,Doench2014_human,Hsu2013,ANPEP
2,AACCGCTGGACCCTGCAGAT,CATGAACCGCTGGACCCTGCAGATGGGCTT,GGG,ANPEP,-0.436401,Doench2014_human,Hsu2013,ANPEP
3,AACGATCTCTTCAGCACATC,CCAGAACGATCTCTTCAGCACATCAGGCAA,AGG,ANPEP,-1.524839,Doench2014_human,Hsu2013,ANPEP
4,AACGATTCCACGCTTTACTT,CGGTAACGATTCCACGCTTTACTTTGGTCC,TGG,ANPEP,-1.079369,Doench2014_human,Hsu2013,ANPEP


## Recreate the fixed split

In [7]:
outer_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SPLIT_SEED,
)

seen_idx, unseen_idx = next(
    outer_splitter.split(
        all_data,
        y=all_data["dataset"],
        groups=all_data["target"],
    )
)

seen_data = all_data.iloc[seen_idx].reset_index(drop=True)
unseen_data = all_data.iloc[unseen_idx].reset_index(drop=True)

inner_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SPLIT_SEED,
)

train_idx, test_idx = next(
    inner_splitter.split(
        seen_data,
        y=seen_data["dataset"],
        groups=seen_data["target"],
    )
)

train_data = seen_data.iloc[train_idx].reset_index(drop=True)
test_data = seen_data.iloc[test_idx].reset_index(drop=True)

assert set(train_data["target"]).isdisjoint(set(test_data["target"]))
assert set(train_data["target"]).isdisjoint(set(unseen_data["target"]))
assert set(test_data["target"]).isdisjoint(set(unseen_data["target"]))

display(pd.DataFrame({
    "subset": ["train", "test", "unseen"],
    "n_rows": [len(train_data), len(test_data), len(unseen_data)],
    "fraction": [
        len(train_data)/len(all_data),
        len(test_data)/len(all_data),
        len(unseen_data)/len(all_data),
    ],
}))


,subset,n_rows,fraction
0,train,32563,0.637403
1,test,8498,0.166344
2,unseen,10026,0.196253


## Prepare and export aligned files

In [9]:
def prepare_export_table(df):
    out = df.copy().reset_index(drop=True)

    out["sgRNA_sequence"] = (
        out["sgRNA Sequence"].astype(str).str.strip().str.upper()
    )
    out["target_context_sequence"] = (
        out["sgRNA Context Sequence"].astype(str).str.strip().str.upper()
    )

    lengths = out["target_context_sequence"].str.len()

    out["target_protospacer_20nt"] = np.where(
        lengths >= 27,
        out["target_context_sequence"].str.slice(4, 24),
        out["sgRNA_sequence"],
    )

    out["pam_sequence"] = np.where(
        lengths >= 27,
        out["target_context_sequence"].str.slice(-6, -3),
        "",
    )

    out["activity"] = pd.to_numeric(
        out["sgRNA Activity"], errors="coerce"
    )

    metadata = [
        c for c in ["dataset", "tracr", "target", "sgRNA Target"]
        if c in out.columns
    ]

    out = out[
        [
            "sgRNA_sequence",
            "target_protospacer_20nt",
            "target_context_sequence",
            "pam_sequence",
            "activity",
        ] + metadata
    ]

    return out.dropna(
        subset=["sgRNA_sequence", "target_context_sequence", "activity"]
    ).reset_index(drop=True)


def save_series(series, path):
    series.to_csv(path, index=False, header=False)


def export_subset(df, subset_name):
    export_df = prepare_export_table(df)
    subset_dir = OUTPUT_DIR / subset_name
    subset_dir.mkdir(parents=True, exist_ok=True)

    save_series(
        export_df["sgRNA_sequence"],
        subset_dir / f"{subset_name}_sgRNA_sequences.txt",
    )
    save_series(
        export_df["target_protospacer_20nt"],
        subset_dir / f"{subset_name}_target_20nt.txt",
    )
    save_series(
        export_df["target_context_sequence"],
        subset_dir / f"{subset_name}_target_context_full.txt",
    )
    save_series(
        export_df["pam_sequence"],
        subset_dir / f"{subset_name}_pam.txt",
    )
    save_series(
        export_df["activity"],
        subset_dir / f"{subset_name}_activity.txt",
    )

    export_df.to_csv(
        subset_dir / f"{subset_name}_complete.tsv",
        sep="\t",
        index=False,
    )

    print(f"{subset_name}: {len(export_df)} rows")
    return export_df


train_export = export_subset(train_data, "train")
test_export = export_subset(test_data, "test")
unseen_export = export_subset(unseen_data, "unseen")


train: 32563 rows
test: 8498 rows
unseen: 10026 rows


## Check sequence lengths and file alignment

In [11]:
def count_lines(path):
    with open(path, "r") as f:
        return sum(1 for line in f if line.strip())


for subset_name, export_df in [
    ("train", train_export),
    ("test", test_export),
    ("unseen", unseen_export),
]:
    print(f"\n{subset_name}")
    print(
        "sgRNA lengths:",
        export_df["sgRNA_sequence"].str.len().value_counts().sort_index().to_dict(),
    )
    print(
        "target 20-nt lengths:",
        export_df["target_protospacer_20nt"].str.len().value_counts().sort_index().to_dict(),
    )
    print(
        "context lengths:",
        export_df["target_context_sequence"].str.len().value_counts().sort_index().to_dict(),
    )

    subset_dir = OUTPUT_DIR / subset_name
    files = [
        subset_dir / f"{subset_name}_sgRNA_sequences.txt",
        subset_dir / f"{subset_name}_target_20nt.txt",
        subset_dir / f"{subset_name}_target_context_full.txt",
        subset_dir / f"{subset_name}_pam.txt",
        subset_dir / f"{subset_name}_activity.txt",
    ]

    counts = [count_lines(path) for path in files]
    print("TXT line counts:", counts)
    assert len(set(counts)) == 1

print("\nAll exported files are aligned.")



train
sgRNA lengths: {20: 32563}
target 20-nt lengths: {20: 32563}
context lengths: {30: 32563}
TXT line counts: [32563, 32563, 32563, 32563, 32563]

test
sgRNA lengths: {20: 8498}
target 20-nt lengths: {20: 8498}
context lengths: {30: 8498}
TXT line counts: [8498, 8498, 8498, 8498, 8498]

unseen
sgRNA lengths: {20: 10026}
target 20-nt lengths: {20: 10026}
context lengths: {30: 10026}
TXT line counts: [10026, 10026, 10026, 10026, 10026]

All exported files are aligned.


In [13]:
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR))


test/test_activity.txt
test/test_complete.tsv
test/test_pam.txt
test/test_sgRNA_sequences.txt
test/test_target_20nt.txt
test/test_target_context_full.txt
train/train_activity.txt
train/train_complete.tsv
train/train_pam.txt
train/train_sgRNA_sequences.txt
train/train_target_20nt.txt
train/train_target_context_full.txt
unseen/unseen_activity.txt
unseen/unseen_complete.tsv
unseen/unseen_pam.txt
unseen/unseen_sgRNA_sequences.txt
unseen/unseen_target_20nt.txt
unseen/unseen_target_context_full.txt
